# Final models and results

After active learning and the labelling of the rest of the pool, all five models were trained once more, from scratch, on all labels. The rules were fixed before training:

1. Five models, 3 seeds each, trained without the 32 training crops that were copies of test crops, and without any crop from a test-2 or test-3 photo.
2. Validation is the original 169 validation crops plus 215 active-learning crops (20%, split by photo).
3. The model is chosen on validation; test 2 confirms.
4. Test 3 is opened once, for the chosen model.

| Model | Validation | Test 2 | Test 2, C / D |
|---|---|---|---|
| Ensemble of the two fine-tuned models | 0.767 | 0.702 | 0.70 / 0.60 |
| DINOv3 ConvNeXt-T, fine-tuned | 0.737 | 0.702 | 0.70 / 0.60 |
| **DINOv3 ViT-S/16, fine-tuned (chosen)** | 0.718 | 0.691 | 0.69 / 0.71 |
| DINOv3 ViT-S/16, frozen | 0.691 | 0.645 | 0.63 / 0.55 |
| SigLIP 2, frozen | 0.664 | 0.655 | 0.69 / 0.52 |
| ConvNeXt-T IN-22k, frozen | 0.592 | 0.615 | 0.67 / 0.57 |

The rule would have picked the ensemble, but the top three were close on validation. The fine-tuned ViT-S was chosen instead, because of its higher C/D mean on test 2 (0.70 vs 0.65) and its lower MAE (0.38), and because it is one model instead of two. The choice was written down before test 3 was opened, and is recorded as a deviation from the plan in `docs/protocol.md`. Because test 2 helped make this choice, only test 3 gives a clean final number.

In [ ]:
# On Colab: install the package and get the configuration files.
!pip -q install "insarag-triage[roboflow] @ git+https://github.com/leeelyehezkel/insarag-damage-triage"
!git clone -q https://github.com/leeelyehezkel/insarag-damage-triage repo 2>/dev/null || true

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from insarag_triage import core as L, analysis as A, active_learning as AL, data as DATA

REPO = Path('repo') if Path('repo').exists() else Path('.')
WORK = Path('work')
dcfg = DATA.load_config(REPO / 'configs' / 'data.yaml')
folders = {s: DATA.download(dcfg, s, WORK / 'roboflow') for s in ['original', 'al_labels', 'test12', 'test3']}
m = DATA.assemble(REPO / 'data' / 'manifests' / 'labels.csv', folders)
pd.crosstab(m.split, m.label, margins=True)

## Train the five models

About 2–3 hours on a Colab T4 in total, mostly the two fine-tuned models. Finished seeds are skipped.

In [ ]:
import dataclasses
RUN = False                       # set True to train
SEEDS = [0, 1, 2]
FINAL = WORK / 'runs' / 'final'
MODELS = [
    L.Config('v3_s_partial_448', backbone='dinov3_vits16', mode='partial', img_size=448, lr_backbone=2e-5),
    L.Config('v3_cnxt_partial_448', backbone='dinov3_convnext_tiny', mode='partial', img_size=448, lr_backbone=5e-5),
    L.Config('v3_s_frozen_224', backbone='dinov3_vits16', mode='frozen', img_size=224),
    L.Config('siglip2_frozen_224', backbone='siglip2_b16', mode='frozen', img_size=224),
    L.Config('cnx_in22k_frozen_224', backbone='convnext_tiny_in22k', mode='frozen', img_size=224),
]
df = pd.concat([m[m.split.isin(['train', 'val'])], m[m.split == 'test1'].assign(split='test')], ignore_index=True)
print(f"train {int((df.split == 'train').sum())} · val {int((df.split == 'val').sum())} · test 1 {int((df.split == 'test').sum())}")
if RUN:
    for c in MODELS:
        for s in SEEDS:
            if not (FINAL / c.name / f'seed{s}' / 'result.json').exists():
                L.run(c, df, s, FINAL / c.name / f'seed{s}')

## Compare on validation and test 2

Each model's P(grade > t) is averaged over its seeds; the ensemble averages the two fine-tuned models.

In [ ]:
EVAL = {'val': m[m.split == 'val'].reset_index(drop=True), 'test2': m[m.split == 'test2'].reset_index(drop=True)}

def cum(c, part):
    f = FINAL / c.name / f'cum_{part}.npy'
    if f.exists():
        return np.load(f)
    seeds = [s for s in SEEDS if (FINAL / c.name / f'seed{s}' / 'result.json').exists()]
    v = np.mean([AL.member_cum_probs(c, FINAL / c.name, s, EVAL[part].path) for s in seeds], axis=0)
    np.save(f, v); return v

CUM = {c.name: {p: cum(c, p) for p in EVAL} for c in MODELS if (FINAL / c.name).exists()}
if {'v3_s_partial_448', 'v3_cnxt_partial_448'} <= set(CUM):
    CUM['ensemble'] = {p: (CUM['v3_s_partial_448'][p] + CUM['v3_cnxt_partial_448'][p]) / 2 for p in EVAL}
rows = []
for name, cm in CUM.items():
    r = {'model': name}
    for p, d in EVAL.items():
        e = L.evaluate(d.y, (cm[p] > 0.5).sum(1))
        r[f'{p} acc'], r[f'{p} MAE'] = e['overall_accuracy_diag_over_6'], e['mae_overall_micro']
        if p == 'test2':
            r['test2 C'], r['test2 D'] = e['per_class_accuracy']['C'], e['per_class_accuracy']['D']
    rows.append(r)
pd.DataFrame(rows).sort_values('val acc', ascending=False).round(3)

## Open test 3, once

Set `CHOICE` before running this cell. The choice is saved first, so it can't be changed after seeing the result. The two duplicate-photo crops of test 3 are already marked `excluded` in the manifest.

Result in the study (DINOv3 ViT-S/16, fine-tuned; 152 gradeable crops):

| Measure | Test 3 |
|---|---|
| Balanced accuracy | 0.73 [0.66–0.80] |
| MAE | 0.29 |
| Within one grade | 97% |
| C / D | 0.72 / 0.87 |
| B | 0.47 [0.20–0.73] (15 crops) |

The most common mistake was C predicted as D (10 of 53). No collapsed building (0 of 67) was predicted as standing; all 11 standing/collapsed errors went the cautious way.

In [ ]:
CHOICE = 'v3_s_partial_448'
choice_file = FINAL / 'choice.txt'
if choice_file.exists() and choice_file.read_text().strip() != CHOICE:
    raise SystemExit(f'test 3 was already opened for {choice_file.read_text().strip()!r}')
choice_file.write_text(CHOICE)

t3 = m[m.split == 'test3'].reset_index(drop=True)
c = next(x for x in MODELS if x.name == CHOICE)
seeds = [s for s in SEEDS if (FINAL / c.name / f'seed{s}' / 'result.json').exists()]
p3 = (np.mean([AL.member_cum_probs(c, FINAL / c.name, s, t3.path) for s in seeds], axis=0) > 0.5).sum(1)
pred3 = pd.DataFrame({'group': t3.photo, 'y_true': t3.y, 'y_pred': p3})
L.plot_confusion(L.evaluate(pred3.y_true, pred3.y_pred), f'{CHOICE} · test 3'); plt.show()
A.bootstrap_ci(pred3).loc[['overall_accuracy_diag_over_6', 'mae_overall_micro', 'within1_balanced']
                          + [f'per_class_accuracy:{g}' for g in L.CLASSES]]

In [ ]:
# Standing (A–C) vs collapsed (D–F); then pairs of neighbouring grades (A–B, C–D, E–F)
side = lambda y: np.where(y <= 2, 'standing (A–C)', 'collapsed (D–F)')
display(pd.crosstab(side(pred3.y_true), side(pred3.y_pred), rownames=['true'], colnames=['predicted']))
grp = lambda y: pd.Series(y).map({0: 'A–B', 1: 'A–B', 2: 'C–D', 3: 'C–D', 4: 'E–F', 5: 'E–F'})
pd.crosstab(grp(pred3.y_true), grp(pred3.y_pred), rownames=['true'], colnames=['predicted'])

## Does more labelled data help?

The chosen model was trained on 25%, 50%, 75% and 100% of the final training set, with one seed per size. The subsets are nested and drawn by photo, so only the amount of data changes. Results from the study:

| Training crops | 379 | 794 | 1,210 | 1,620 |
|---|---|---|---|---|
| Validation balanced accuracy | 0.691 | 0.719 | 0.726 | 0.731 |
| Validation MAE | 0.417 | 0.383 | 0.365 | 0.352 |
| Test 2 balanced accuracy | 0.664 | 0.697 | 0.681 | 0.706 |

- **Most of the gain comes by about 800 crops.** Doubling to 1,620 added about 0.01 on validation. Test 2 moves within its ±0.07 interval.
- **By grade,** D gained most (0.62 → 0.74 on test 2); C and B stayed flat.
- **The kind of data matters more than the amount.** The same model trained on the 723 original crops alone scored 0.573 on test 2, below the 379-crop subset of the final training set (0.664). The final set contains crops from the same kind of photos as test 2, at full size, labelled under the current rules.

In [ ]:
import math
RUN_CURVE = False
FRACS = [0.25, 0.5, 0.75, 1.0]
c = next(x for x in MODELS if x.name == CHOICE)
train = df[df.split == 'train']
order = np.random.default_rng(42).permutation(np.array(sorted(train.photo.unique())))
def subset(f):
    keep = set(order[:math.ceil(f * len(order))])
    return pd.concat([df[df.split != 'train'], train[train.photo.isin(keep)]], ignore_index=True)
rows = []
for f in FRACS:
    out = (FINAL / c.name if f == 1.0 else WORK / 'runs' / 'curve' / f'{c.name}_f{int(f * 100)}') / 'seed0'
    if RUN_CURVE and not (out / 'result.json').exists():
        L.run(c, subset(f), 0, out)
    if (out / 'result.json').exists():
        r = json.loads((out / 'result.json').read_text())
        p2 = (AL.member_cum_probs(c, out.parent, 0, EVAL['test2'].path) > 0.5).sum(1)
        e2 = L.evaluate(EVAL['test2'].y, p2)
        rows.append(dict(crops=int((subset(f).split == 'train').sum()), val_acc=r['val']['overall_accuracy_diag_over_6'],
                         val_mae=r['val']['mae_overall_micro'], test2_acc=e2['overall_accuracy_diag_over_6'], test2_mae=e2['mae_overall_micro']))
curve = pd.DataFrame(rows)
if len(curve) > 1:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(curve.crops, curve.val_acc, 's--', label='validation'); ax[0].plot(curve.crops, curve.test2_acc, 'o-', label='test 2')
    ax[1].plot(curve.crops, curve.val_mae, 's--', label='validation'); ax[1].plot(curve.crops, curve.test2_mae, 'o-', label='test 2')
    ax[0].set(xlabel='training crops', ylabel='balanced accuracy', title='More labelled crops: balanced accuracy')
    ax[1].set(xlabel='training crops', ylabel='MAE (grades)', title='More labelled crops: error in grades')
    for a in ax: a.grid(alpha=0.3); a.legend()
    plt.tight_layout(); plt.show()
curve.round(3)